# Phase 1 baseline: raw dead reckoning vs. EKF

**Note:** IO-VNBD is not bundled in this repository (see `data/raw/io_vnbd/`,
which is gitignored). Until it's downloaded, this notebook validates the
pipeline end-to-end on a synthetic trajectory generated by
`src/synthetic_data.py`. Swap in a real IO-VNBD sequence via
`src/data_loader.load_sequence` once the dataset is present
-- the downstream EKF/metrics code is identical either way.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from src.synthetic_data import generate_synthetic_drive
from src.preprocess import simulate_gnss_outage, latlon_to_enu
from src.filters import DeadReckoningEKF
from src.visualizer import plot_trajectory_comparison

gt, imu, gnss, ref_lat, ref_lon = generate_synthetic_drive(duration_s=90, hz=50, seed=1)
print(f"{len(imu)} IMU samples, {len(gnss)} GNSS fixes, {gt['east'].iloc[-1]:.0f}m east / {gt['north'].iloc[-1]:.0f}m north final ground-truth offset")

In [ ]:
def raw_dead_reckoning(imu_df, dt):
    n = len(imu_df)
    east = np.zeros(n); north = np.zeros(n)
    ve = vn = heading = 0.0
    for i in range(1, n):
        a = float(imu_df['accel_x'].iloc[i])
        wz = float(imu_df['gyro_z'].iloc[i])
        heading += wz * dt
        ve += a * np.cos(heading) * dt
        vn += a * np.sin(heading) * dt
        east[i] = east[i-1] + ve * dt
        north[i] = north[i-1] + vn * dt
    return east, north

dt = 1.0 / 50
raw_east, raw_north = raw_dead_reckoning(imu, dt)

In [ ]:
# Simulate a 30s GNSS outage from t=30s to t=60s, run the EKF through it.
gnss_outage = simulate_gnss_outage(gnss, start_sec=30, duration_sec=30)
east_gnss, north_gnss = latlon_to_enu(gnss_outage['gnss_lat'].to_numpy(), gnss_outage['gnss_lon'].to_numpy(), ref_lat, ref_lon)

ekf = DeadReckoningEKF(np.zeros(7), np.eye(7) * 0.1)
gnss_times = gnss_outage['timestamp'].to_numpy()
gnss_ptr = 0
est_east = np.zeros(len(imu)); est_north = np.zeros(len(imu))
for i in range(len(imu)):
    t = imu['timestamp'].iloc[i]
    a = float(imu['accel_x'].iloc[i]); wz = float(imu['gyro_z'].iloc[i])
    if i == 0:
        continue
    ekf.predict(a, wz, dt)
    while gnss_ptr < len(gnss_times) and gnss_times[gnss_ptr] <= t:
        ekf.correct_gnss([east_gnss[gnss_ptr], north_gnss[gnss_ptr]], R_gnss=np.diag([9., 9.]))
        gnss_ptr += 1
    est_east[i], est_north[i] = ekf.x[0], ekf.x[1]

In [ ]:
gt_east = gt['east'].to_numpy(); gt_north = gt['north'].to_numpy()
distance_travelled = np.sum(np.hypot(np.diff(gt_east), np.diff(gt_north)))
raw_err = np.hypot(raw_east[-1] - gt_east[-1], raw_north[-1] - gt_north[-1])
ekf_err = np.hypot(est_east[-1] - gt_east[-1], est_north[-1] - gt_north[-1])

print(f"Distance travelled:  {distance_travelled:.1f} m")
print(f"Raw DR final error:  {raw_err:.1f} m  ({100*raw_err/distance_travelled:.1f}% drift)")
print(f"EKF final error:     {ekf_err:.1f} m  ({100*ekf_err/distance_travelled:.1f}% drift)")
print()
print("NOTE: this baseline EKF corrects POSITION from GNSS only -- it does not")
print("yet fuse heading from GNSS course-over-ground or map geometry. Ground")
print("truth includes a ~90-degree turn late in the drive that neither raw DR")
print("nor this baseline EKF tracks correctly. This is expected at this phase")
print("and is exactly what ZUPT + non-holonomic constraints + AI speed + ")
print("map-matching (Phases 2-5) are designed to fix -- not a bug to hide.")

In [ ]:
gt_arr = np.column_stack([gt_east, gt_north])
raw_arr = np.column_stack([raw_east, raw_north])
ekf_arr = np.column_stack([est_east, est_north])
plot_trajectory_comparison(gt_arr, raw_arr, ekf_arr, ekf_arr, save_path='../results/plots/synthetic_baseline_comparison.png')